<a href="https://colab.research.google.com/github/Valbu11/Neurovault-Brain-Data-Analysis/blob/main/notebooks/02_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Notebook 02 — Data Cleaning

**Goal:** Clean and transform the raw NeuroVault data so it is ready for analysis and visualization.

**What you will learn:**
- How to handle missing values (nulls)
- How to fix data types (dates, numbers, strings)
- How to select only useful columns
- How to create new columns from existing ones
- How to save clean data for the next notebooks

**Input:** `data/raw/collections.csv` + `data/raw/images.csv`  
**Output:** `data/processed/collections_clean.csv` + `data/processed/images_clean.csv`

---
## 1. Import libraries

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 50)

print('Libraries imported ✅')

Libraries imported ✅


---
## 2. Load raw data

In [6]:
df_collections = pd.read_csv('data/raw/collections.csv', low_memory=False)
df_images = pd.read_csv('data/raw/images.csv', low_memory=False)

print(f'Collections: {df_collections.shape[0]:,} rows × {df_collections.shape[1]} columns')
print(f'Images:      {df_images.shape[0]:,} rows × {df_images.shape[1]} columns')

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/collections.csv'

---
## 3. Select useful columns

Collections has 107 columns — most are empty or irrelevant. We keep only the ones that tell us something meaningful.

In [ ]:
COLLECTION_COLS = [
    'id',
    'name',
    'description',
    'number_of_images',
    'add_date',
    'modify_date',
    'DOI',
    'authors',
    'paper_url',
    'owner',
    'full_dataset_url',
    'subjects',
    'scanner_make',
    'scanner_model',
    'field_strength',
]

IMAGE_COLS = [
    'id',
    'name',
    'modality',
    'map_type',
    'add_date',
    'modify_date',
    'collection',
    'description',
    'number_of_subjects',
    'brain_coverage',
    'is_valid',
]

# Keep only columns that exist in the dataframe
col_cols = [c for c in COLLECTION_COLS if c in df_collections.columns]
img_cols = [c for c in IMAGE_COLS if c in df_images.columns]

df_c = df_collections[col_cols].copy()
df_i = df_images[img_cols].copy()

print(f'Collections: {df_c.shape[1]} columns selected (from 107)')
print(f'Images:      {df_i.shape[1]} columns selected (from 66)')
df_c.head(3)

---
## 4. Fix data types — dates

Dates come as strings. We convert them to datetime so we can extract year, month, etc.

In [ ]:
# Convert dates with UTC to avoid mixed timezone errors
for df, name in [(df_c, 'collections'), (df_i, 'images')]:
    for col in ['add_date', 'modify_date']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
            print(f'  {name}.{col} → {df[col].dtype}')

print('\nDates fixed ✅')

---
## 5. Create new columns

From the dates we extract `year` and `month` — very useful for trend analysis.

In [ ]:
# Collections
df_c['year']  = df_c['add_date'].dt.year
df_c['month'] = df_c['add_date'].dt.month
df_c['has_doi'] = df_c['DOI'].notna().astype(int)       # 1 if has DOI, 0 if not
df_c['has_authors'] = df_c['authors'].notna().astype(int)

# Images
df_i['year']  = df_i['add_date'].dt.year
df_i['month'] = df_i['add_date'].dt.month

print('New columns created:')
print('  collections → year, month, has_doi, has_authors')
print('  images      → year, month')
print()
print(df_c[['id', 'name', 'year', 'month', 'has_doi', 'has_authors']].head(5))

---
## 6. Handle missing values

We don't delete rows with nulls — we fill them with meaningful defaults.

In [ ]:
# Collections — fill text nulls with 'Unknown'
text_cols = ['description', 'DOI', 'authors', 'paper_url', 'scanner_make', 'scanner_model']
for col in text_cols:
    if col in df_c.columns:
        df_c[col] = df_c[col].fillna('Unknown')

# Images — fill modality nulls with 'Unknown'
df_i['modality'] = df_i['modality'].fillna('Unknown')
df_i['map_type'] = df_i['map_type'].fillna('Unknown')

print('Null values after cleaning:')
print('Collections:')
remaining = df_c.isnull().sum()
print(remaining[remaining > 0] if remaining.sum() > 0 else '  No nulls ✅')
print('\nImages:')
remaining_i = df_i.isnull().sum()
print(remaining_i[remaining_i > 0] if remaining_i.sum() > 0 else '  No nulls ✅')

---
## 7. Quick EDA — first insights

A fast look at the clean data before saving.

In [ ]:
print('=== STUDIES PER YEAR ===')
print(df_c.groupby('year')['id'].count().to_string())

print('\n=== IMAGE MODALITY ===')
print(df_i['modality'].value_counts().to_string())

print('\n=== MAP TYPES ===')
print(df_i['map_type'].value_counts().head(8).to_string())

print('\n=== STUDIES WITH DOI ===')
doi_pct = df_c['has_doi'].mean() * 100
print(f'  With DOI:    {df_c["has_doi"].sum():,} ({doi_pct:.1f}%)')
print(f'  Without DOI: {(~df_c["has_doi"].astype(bool)).sum():,} ({100-doi_pct:.1f}%)')

---
## 8. Save clean data

In [ ]:
os.makedirs('data/processed', exist_ok=True)

df_c.to_csv('data/processed/collections_clean.csv', index=False)
df_i.to_csv('data/processed/images_clean.csv', index=False)

print('Clean data saved ✅')
print(f'  data/processed/collections_clean.csv → {df_c.shape[0]:,} rows × {df_c.shape[1]} columns')
print(f'  data/processed/images_clean.csv      → {df_i.shape[0]:,} rows × {df_i.shape[1]} columns')
print()
print('Next step → 03_sql_analysis.ipynb 🚀')

---
## ✅ What we accomplished

- Reduced columns from 107 → 15 (collections) and 66 → 11 (images)
- Fixed date types and extracted year/month
- Created useful indicator columns: `has_doi`, `has_authors`
- Filled missing values with meaningful defaults
- Saved clean datasets ready for SQL and visualization

**Next notebook:** `03_sql_analysis.ipynb` — SQL queries with SQLite to answer business questions.